In [1]:
import os
import sys
sys.path.append('..')

from src.rag_pipeline import build_vectorstore, rag_pipeline

c:\Users\liauw\Desktop\Sputnik\2025-26\Courses\block-6\575-nlp\DSCI_575_project_cliauwyt_cea\env\Lib\site-packages\langchain_core\_api\deprecation.py:25: UserWarning: Core Pydantic V1 functionality isn't compatible with Python 3.14 or greater.
  from pydantic.v1.fields import FieldInfo as FieldInfoV1


# Choose a model

In [2]:
from transformers import pipeline
generator = pipeline(
    task="text-generation",
    model="Qwen/Qwen3.5-0.8B"
)

The fast path is not available because one of the required library is not installed. Falling back to torch implementation. To install follow https://github.com/fla-org/flash-linear-attention#installation and https://github.com/Dao-AILab/causal-conv1d


Loading weights:   0%|          | 0/320 [00:00<?, ?it/s]

# RAG Semantic

In [3]:
from langchain_community.vectorstores import FAISS
from langchain_huggingface import HuggingFaceEmbeddings

embeddings = HuggingFaceEmbeddings(
    model_name="sentence-transformers/all-MiniLM-L6-v2"
)

corpus_path = '../data/processed/preprocessed_corpus.csv'
vector_path = "../data/processed/vector_store"

if os.path.exists(vector_path):
    vectorstore = FAISS.load_local(
        vector_path, embeddings, allow_dangerous_deserialization=True
    )
else:
    build_vectorstore(corpus_path, vector_path, embeddings)
    print(f"Saved vector store to {vector_path}")

Loading weights:   0%|          | 0/103 [00:00<?, ?it/s]

BertModel LOAD REPORT from: sentence-transformers/all-MiniLM-L6-v2
Key                     | Status     |  | 
------------------------+------------+--+-
embeddings.position_ids | UNEXPECTED |  | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not ok if you expect identical arch.


In [4]:
rag = rag_pipeline(vectorstore, "what is the best soap", generator)
print(rag)

Both `max_new_tokens` (=256) and `max_length`(=20) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)


Human: You are a helpful Amazon shopping assistant.
    Answer the question using ONLY the following context (real product reviews + metadata).
    Always cite the product ASIN when possible.

    Customer Reviews: Product ASIN: B0716PQVP2
Title: Dealglad 10Pcs Double Layer Exfoliating Mesh Soap Saver Pouch Bubble Foam Net Handmade Soap Mesh Bag Body Facial Cleaning Tool
Rating: 5.0/5.0
Review: Must have with bars of soap ! You will love !


Product ASIN: B08DV37PZV
Title: Palmolive Ultra Original Dish Liquid, 102 fl. oz. - 2 Pack
Rating: 5.0/5.0
Review: That "blue" dish soap is more difficult to rinse off.  I like Palmolive because it cleans well and rinses off easily.


Product ASIN: B001HDZT7I
Title: Travelon Hand Soap Toiletry Sheets, 50-Count
Rating: 3.0/5.0
Review: Although this had decent reviews when I researched I learned very quickly that once these are wet they are not usable. The sheet size maybe can cover hand washing but they clump up and there isn't a lather like with ba